In [ ]:
"""
BSC — Mapa de Causalidade com Interdependências Quantificáveis
═══════════════════════════════════════════════════════════════
PASSO A: Simulação do Dataset

Objetivo: gerar um dataset de N observações que respeita a estrutura
causal do BSC. As correlações entre variáveis emergem das relações
causais — não são impostas artificialmente.

Cada observação representa um período ou unidade organizacional.
"""

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── Reprodutibilidade ────────────────────────────────────────────────────────
# Fixar a semente garante que você obtém os mesmos dados toda vez.
# Troque o valor para gerar amostras diferentes.
np.random.seed(42)

N = 200  # número de observações

print("=" * 60)
print("PASSO A — Simulação do Dataset BSC")
print("=" * 60)

# ════════════════════════════════════════════════════════════════
# BLOCO 1 — Perspectiva de Aprendizado & Crescimento
# Variáveis raiz: não têm predecessores no grafo.
# São geradas como puro ruído gaussiano com parâmetros realistas.
# ════════════════════════════════════════════════════════════════

A1 = np.random.normal(loc=32, scale=6, size=N)
#    └─ Horas de Treinamento: média 32h, desvio 6h

A2 = np.random.normal(loc=850, scale=150, size=N)
#    └─ Investimento em Tecnologia: média R$850k, desvio R$150k

print("\n[A] Aprendizado & Crescimento")
print(f"    A1 (Horas Treinamento)   → média: {A1.mean():.1f}h  | dp: {A1.std():.1f}h")
print(
    f"    A2 (Invest. Tecnologia)  → média: R${A2.mean():.0f}k | dp: R${A2.std():.0f}k"
)


PASSO A — Simulação do Dataset BSC

[A] Aprendizado & Crescimento
    A1 (Horas Treinamento)   → média: 31.8h  | dp: 5.6h
    A2 (Invest. Tecnologia)  → média: R$863k | dp: R$148k


In [5]:
# ════════════════════════════════════════════════════════════════
# PADRONIZAÇÃO das variáveis de entrada
#
# Por quê? A1 está em horas (escala ~30) e A2 em R$k (escala ~850).
# Se usarmos as escalas brutas, os β não serão comparáveis.
# Padronizar (z-score) coloca tudo em desvios padrão.
#
# z = (x - média) / desvio_padrão
# Resultado: média 0, desvio padrão 1
# ════════════════════════════════════════════════════════════════

def zscale(x):
    return (x - x.mean()) / x.std()


A1_s = zscale(A1)
A2_s = zscale(A2)



In [6]:
# ════════════════════════════════════════════════════════════════
# BLOCO 2 — Perspectiva de Processos Internos
# ════════════════════════════════════════════════════════════════

# P1 (Eficiência Operacional) ← A1, A2
# Interpretação dos β:
#   +0.38: mais treinamento → mais eficiência
#   +0.31: mais tecnologia  → mais eficiência
# O ruído (scale=0.04) representa fatores não modelados
P1 = (
    0.74  # intercepto (nível base de eficiência)
    + 0.38 * A1_s  # efeito do treinamento
    + 0.31 * A2_s  # efeito da tecnologia
    + np.random.normal(0, 0.04, N)
)  # ruído
P1 = np.clip(P1, 0.40, 0.99)  # eficiência é um índice [0,1]

# P2 (Taxa de Retrabalho) ← P1
# β NEGATIVO: mais eficiência → MENOS retrabalho (relação inversa)
P2 = (
    0.35
    - 0.55 * zscale(P1)  # β negativo = relação inversa
    + np.random.normal(0, 0.02, N)
)
P2 = np.clip(P2, 0.01, 0.50)

print("\n[P] Processos Internos")
print(f"    P1 (Eficiência Operacional) → média: {P1.mean():.3f} | dp: {P1.std():.3f}")
print(f"    P2 (Taxa de Retrabalho)     → média: {P2.mean():.3f} | dp: {P2.std():.3f}")



[P] Processos Internos
    P1 (Eficiência Operacional) → média: 0.727 | dp: 0.241
    P2 (Taxa de Retrabalho)     → média: 0.257 | dp: 0.224


In [7]:
# ════════════════════════════════════════════════════════════════
# BLOCO 3 — Perspectiva de Clientes
# ════════════════════════════════════════════════════════════════

# C1 (Satisfação do Cliente) ← P1, P2
# P1 tem β positivo (eficiência melhora satisfação)
# P2 tem β negativo (retrabalho prejudica satisfação)
C1 = 0.30 + 0.44 * zscale(P1) - 0.28 * zscale(P2) + np.random.normal(0, 0.03, N)
C1 = np.clip(C1, 0.30, 0.99)

# C2 (Taxa de Retenção) ← C1
# Clientes satisfeitos ficam → relação positiva forte
C2 = 0.20 + 0.62 * zscale(C1) + np.random.normal(0, 0.03, N)
C2 = np.clip(C2, 0.30, 0.99)

print("\n[C] Clientes")
print(f"    C1 (Satisfação do Cliente) → média: {C1.mean():.3f} | dp: {C1.std():.3f}")
print(f"    C2 (Taxa de Retenção)      → média: {C2.mean():.3f} | dp: {C2.std():.3f}")


[C] Clientes
    C1 (Satisfação do Cliente) → média: 0.596 | dp: 0.326
    C2 (Taxa de Retenção)      → média: 0.557 | dp: 0.311


In [8]:
# ════════════════════════════════════════════════════════════════
# BLOCO 4 — Perspectiva Financeira
# ════════════════════════════════════════════════════════════════

# F1 (Receita) ← C2, P1
# Retenção de clientes e eficiência impactam receita
# Escala grande: R$ mil
F1 = 1500 + 1800 * zscale(C2) + 900 * zscale(P1) + np.random.normal(0, 200, N)
F1 = np.clip(F1, 500, 8000)

# F2 (Margem Operacional) ← F1
# Mais receita → mais margem (economias de escala)
F2 = 0.05 + 0.71 * zscale(F1) + np.random.normal(0, 0.03, N)
F2 = np.clip(F2, 0.01, 0.60)

print("\n[F] Financeiro")
print(
    f"    F1 (Receita)              → média: R${F1.mean():.0f}k | dp: R${F1.std():.0f}k"
)
print(f"    F2 (Margem Operacional)   → média: {F2.mean():.3f} | dp: {F2.std():.3f}")


[F] Financeiro
    F1 (Receita)              → média: R$2154k | dp: R$2017k
    F2 (Margem Operacional)   → média: 0.240 | dp: 0.285


In [9]:
# ════════════════════════════════════════════════════════════════
# BLOCO 5 — Montar e inspecionar o DataFrame
# ════════════════════════════════════════════════════════════════

df = pd.DataFrame(
    {
        "A1": A1,
        "A2": A2,  # Aprendizado
        "P1": P1,
        "P2": P2,  # Processos
        "C1": C1,
        "C2": C2,  # Clientes
        "F1": F1,
        "F2": F2,  # Financeiro
    }
)

print(f"\n{'=' * 60}")
print(f"Dataset gerado: {df.shape[0]} observações × {df.shape[1]} variáveis")
print(f"{'=' * 60}")
print("\nEstatísticas descritivas:")
print(df.describe().round(3))


Dataset gerado: 200 observações × 8 variáveis

Estatísticas descritivas:
            A1        A2       P1       P2       C1       C2        F1  \
count  200.000   200.000  200.000  200.000  200.000  200.000   200.000   
mean    31.755   862.880    0.727    0.257    0.596    0.557  2153.552   
std      5.586   148.051    0.241    0.225    0.326    0.312  2021.873   
min     16.282   363.810    0.400    0.010    0.300    0.300   500.000   
25%     27.769   759.124    0.406    0.010    0.300    0.300   500.000   
50%     31.975   861.826    0.757    0.283    0.317    0.300   500.000   
75%     35.005   953.085    0.990    0.500    0.990    0.937  4580.325   
max     48.321  1427.910    0.990    0.500    0.990    0.990  5265.751   

            F2  
count  200.000  
mean     0.240  
std      0.285  
min      0.010  
25%      0.010  
50%      0.010  
75%      0.600  
max      0.600  


In [10]:
# ════════════════════════════════════════════════════════════════
# BLOCO 6 — Verificação: a matriz de correlação
#
# Se a simulação está correta, devemos observar:
#   • A1 e A2 correlacionados positivamente com P1
#   • P2 correlacionado NEGATIVAMENTE com P1 (relação inversa)
#   • C1 e C2 correlacionados positivamente com P1
#   • F1 e F2 correlacionados positivamente com C2 e P1
# ════════════════════════════════════════════════════════════════

print("\nMatriz de Correlação (valores esperados vs. observados):")
corr = df.corr().round(3)
print(corr)



Matriz de Correlação (valores esperados vs. observados):
       A1     A2     P1     P2     C1     C2     F1     F2
A1  1.000  0.095  0.723 -0.669  0.639  0.622  0.622  0.616
A2  0.095  1.000  0.603 -0.559  0.535  0.515  0.521  0.510
P1  0.723  0.603  1.000 -0.954  0.899  0.858  0.855  0.845
P2 -0.669 -0.559 -0.954  1.000 -0.963 -0.908 -0.901 -0.890
C1  0.639  0.535  0.899 -0.963  1.000  0.976  0.971  0.963
C2  0.622  0.515  0.858 -0.908  0.976  1.000  0.997  0.993
F1  0.622  0.521  0.855 -0.901  0.971  0.997  1.000  0.992
F2  0.616  0.510  0.845 -0.890  0.963  0.993  0.992  1.000


In [11]:
# ════════════════════════════════════════════════════════════════
# BLOCO 7 — Visualização: 3 painéis diagnósticos
# ════════════════════════════════════════════════════════════════

DARK = "#070910"
DARK2 = "#0d1017"
DARK3 = "#111620"
BORDER = "#1e2535"
TEXT = "#e2e8f0"
MUTED = "#4a5568"

COLORS = {
    "A": "#3b82f6",  # Aprendizado — azul
    "P": "#8b5cf6",  # Processos   — roxo
    "C": "#10b981",  # Clientes    — verde
    "F": "#f59e0b",  # Financeiro  — âmbar
}